# SimpleTransformers Sentences Classification

https://colab.research.google.com/drive/1klGQVlalu7WAQZOhglJeaVoZQP6EwA1Q#scrollTo=5426e236

In [ ]:
import pandas as pd
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer , AutoModelForSequenceClassification

## Import Dataset

In [ ]:
train_df = pd.read_csv("train_data.csv").drop(["ID"],axis=1)
test_df = pd.read_csv("test_data.csv").drop(["ID"],axis=1)

In [ ]:
train_df.rename(columns={"Review":"text","Rating":"label"},inplace=True)
test_df.rename(columns={"Review":"text"},inplace=True)
train_df

In [ ]:
train_df["label"] = train_df["label"] - 1

In [ ]:
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)
train_ds

## Computation Matrix

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)  # Get predicted class indices
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

## Load Model

In [ ]:
checkpoint = "/home/ai5039/.cache/huggingface/hub/models--FacebookAI--xlm-roberta-base/snapshots/e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=5)

## Preprocess Dataset

In [ ]:
# Map preprocessing
def preprocess_function(examples):
   return tokenizer(examples["text"], truncation=True)

tokenized_train = train_ds.map(preprocess_function, batched=True)

# Train and Eval Split
tokenized_train = tokenized_train.train_test_split(test_size=0.2)

## Training Model

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import TrainingArguments, Trainer

# Setup Training Argument
training_args = TrainingArguments(
    output_dir="./output",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    dataloader_pin_memory =True,
    dataloader_num_workers = 8,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
    fp16 = True,
    save_steps = None
)

# Train Model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train["train"],
    eval_dataset=tokenized_train["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## Inferences Model

In [ ]:
tokenized_test = test_ds.map(preprocess_function, batched=True)
logits = trainer.predict(tokenized_test)

In [ ]:
answer = logits.predictions.argmax(axis=-1) +1
answer = answer.reshape(-1,1)
answer

In [ ]:
submission = pd.read_csv("submit.csv")
submission["Rating"] = answer
submission.to_csv("submission.csv",index=False)